### Create a CDC Connection

#### Step-by-Step Guide to Configure Change Data Capture Between an RDS MSSQL Server and Databricks

This document serves as a guide to implement Change Data Capture from an AWS RDS SQL Server database to a Databricks account deployed in AWS. The main challenge is that the resources are deployed in different VPCs, so multiple networking steps are necessary to establish the connection. Without further ado, these are the steps covered in this guide:

- RDS Database configuration
- Security group rules
- VPC peering and route tables
- TLS/SSL connection settings

### RDS Database Configuration

This section assumes there is no existing database. If you need to deploy one, do so from the AWS Console. This is for demonstration purposes, so you can choose any instance according to your needs (testing, learning, or production), but make sure it is a **MSSQL Server** instance.

![image_1779979920580.png](./image_1779979920580.png "image_1779979920580.png")

Also, make sure the instance is publicly accessible *if* you want to connect from a local desktop, or not publicly accessible if you are going to connect from an EC2 instance in the same VPC.

Once the instance is running, create a database, a schema, and a table. If you already have your database configured, skip this step.

Enable the RDS security group to permit connections from your IP using **TCP** on port **1433**, as shown in the image:

![image_1779987069870.png](./image_1779987069870.png "image_1779987069870.png")

_Ignore the first line for now._ Just add your IP to the security group so you can establish connections to the database from outside the VPC.

In SQL Server, run these commands:

```sql
-- Activate CDC in the database
ALTER DATABASE [database_name] SET CHANGE_TRACKING = ON (CHANGE_RETENTION = 14 DAYS, AUTO_CLEANUP = ON);

-- Set a primary key for each table you are going to track with CDC
ALTER TABLE [database_name].[schema].[table_name] ADD CONSTRAINT [pk_name] PRIMARY KEY CLUSTERED ([column_names]);

-- Enable change tracking on the tables
ALTER TABLE [database_name].[schema].[table_name] ENABLE CHANGE_TRACKING;
```

An alternate version to accomplish this:

```sql
-- Enable CDC on the database
EXEC msdb.dbo.rds_cdc_enable_db '[database_name]'

-- Enable CDC on tables
EXEC sys.sp_cdc_enable_table
@source_schema = N'[schema_name]',
@source_name = N'[table_name]',
@role_name = NULL,
@supports_net_changes = 1;
```

Once that is complete, you can close the connection and change the "Publicly accessible" option on the RDS instance. This will block access to the RDS instance from your local machine, but you can modify that option later if required (_or connect from a Lambda or EC2 instance in the same VPC_).

   
### Networking Problem and How to Fix It

For context, when you create a Databricks account using CloudFormation, it deploys all the Databricks infrastructure inside a single VPC. The RDS database, on the other hand, lives in a different VPC with its own networking rules and security groups. We need a way to allow traffic between Databricks and MSSQL Server.

![image_1779991705396.png](./image_1779991705396.png "image_1779991705396.png")

To fix this, we need to create a networking connection between the two (for now) isolated resources.

**First: VPC Peering**

A VPC peering connection is a method that allows communication between two different VPCs. To create one, in the AWS Console go to **VPC** and select **Peering Connections** from the left menu, then click **Create VPC Peering**.

In this case, the requester VPC (the source of the request) is the Databricks VPC with **CIDR 10.53.0.0/16** (meaning all IPs from Databricks start with "10.53"). The accepter VPC (the target) is the RDS VPC with **CIDR 172.31.0.0/16**. Once created, verify that its status is **Active**.

![image_1780004450087.png](./image_1780004450087.png "image_1780004450087.png")

This step establishes a path for traffic to travel from Databricks to the RDS instance.

**Second: Configure the RDS Security Group**

Remember from the first section, the line I told you to ignore for now? This is where it gets applied. Create a rule in the RDS security group that allows **inbound** traffic from the Databricks CIDR on port **1433** using **TCP**.

**Third: Configure the Databricks Security Group**

This can be trickier because you need to locate all the security groups associated with your Databricks account. In the AWS Console, filter them by your Databricks **VPC ID** and add **outbound** rules for **TCP** on port **1433** to the RDS CIDR (**172.31.0.0/16**).

**Fourth: Edit Route Tables**

Select the route table associated with Databricks and add a VPC peering route as shown in the image:

![image_1780025671053.png](./image_1780025671053.png "image_1780025671053.png")

Analogously, repeat the process for the RDS route table: add the Databricks CIDR to the VPC peering connection. After this, both resources should be able to see each other.

_Check that your VPCs have DNS resolution enabled, as they will need to resolve the RDS host to an IP address._

**Fifth: Test Your Connection from Databricks**

To verify that your database is reachable from Databricks, copy the host URL and test it from a Databricks notebook using the following command:

In [0]:
%sh
nc -zv lakeflow-demo.cf7dd9ibvy2z.us-east-1.rds.amazonaws.com 1433

   
Notice that the URL resolved to 172.31.75.36 and the port is *open*. This confirms that the networking is properly configured, and we can now set up the Databricks configuration for CDC.

## Set Up the Connection from Databricks Using Lakeflow Connect

In the Databricks console, navigate to the left menu and start a **Data Ingestion** pipeline for a SQL Server database. The process is straightforward and can be completed by following the on-screen wizard. Here are a few things to keep in mind:

- You will need the database URL from the RDS console and the credentials (username and password) to create a Catalog external connection.
![image_1780027509513.png](./image_1780027509513.png "image_1780027509513.png")
- If this is the first time you are configuring CDC, you will need to run two scripts provided by Databricks. The first script (utility script) creates stored procedures in the database, and the second script grants your user the permissions needed to enable tracking and capture changed data. The scripts will be displayed during the wizard if the database requires them. The final step in the wizard validates that those scripts were executed successfully.

If you followed the steps correctly, you will see that the pipeline generates two jobs:

![image_1780027950590.png](./image_1780027950590.png "image_1780027950590.png")

These jobs extract data from your database and capture changes into the catalog and schema you configured. In my case, these are the resulting tables:

![image_1780028099208.png](./image_1780028099208.png "image_1780028099208.png")


## Conclusion

Congratulations! You have successfully configured a CDC pipeline from an RDS SQL Server instance to Databricks. While the process may seem complex at first, the real challenge lies in establishing the network connectivity between the two VPCs. Once you verify that both services can communicate (using a simple `nc` test), the remaining configuration is guided by the Lakeflow Connect wizard.

To summarize, the key steps are:
1. Enable CDC on your SQL Server database and tables.
2. Configure VPC peering, security groups, and route tables so traffic can flow between RDS and Databricks.
3. Validate connectivity from a Databricks notebook.
4. Use the Lakeflow Connect wizard to create the ingestion pipeline.

Special thanks to [this video](https://www.youtube.com/watch?v=gFAnlTM-3Zo&t=750s) for laying the foundation for this guide.